In [9]:
from pathlib import Path
import requests

def download_one_file_from_url(year: int, month: int) -> Path:
    """Downloads a single file from the specified URL and saves it to the local filesystem."""
    
    URL = f'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month:02d}.parquet'
    response = requests.get(URL)
    
    if response.status_code == 200:
        path = f'../data/raw/rides_{year}-{month:02d}.parquet'
        open(path, 'wb').write(response.content)
        return path
    else:
        raise Exception(f'Failed to download file from {URL}. Status code: {response.status_code}')


In [10]:
download_one_file_from_url(year=2026, month=5)

'../data/raw/rides_2026-05.parquet'

In [11]:
import pandas as pd

rides = pd.read_parquet('../data/raw/rides_2026-05.parquet')
rides.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-05-01 00:04:59,2026-05-01 00:32:48,1.0,7.51,1.0,N,138,37,2,35.9,6.00,0.5,0.00,0.0,1.0,45.40,0.0,2.0,0.00
1,2,2026-05-01 00:37:05,2026-05-01 00:59:47,1.0,6.14,1.0,N,138,237,1,27.5,6.00,0.5,9.56,0.0,1.0,49.81,2.5,2.0,0.75
2,1,2026-05-01 00:34:05,2026-05-01 00:54:02,1.0,2.40,1.0,N,249,232,1,19.1,4.25,0.5,3.73,0.0,1.0,28.58,2.5,0.0,0.75
3,1,2026-05-01 00:55:07,2026-05-01 01:02:43,0.0,1.20,1.0,N,232,114,1,9.3,4.25,0.5,3.00,0.0,1.0,18.05,2.5,0.0,0.75
4,7,2026-05-01 00:44:13,2026-05-01 00:44:13,2.0,0.86,1.0,N,140,237,1,7.2,0.00,0.5,2.44,0.0,1.0,14.64,2.5,0.0,0.00


In [12]:
rides = rides[['tpep_pickup_datetime', 'PULocationID']]

In [13]:
rides.rename(columns={
    'tpep_pickup_datetime': 'pickup_datetime',
    'PULocationID': 'pickup_location_id'
}, inplace=True)

rides.head(10)

,pickup_datetime,pickup_location_id
0,2026-05-01 00:04:59,138
1,2026-05-01 00:37:05,138
2,2026-05-01 00:34:05,249
3,2026-05-01 00:55:07,232
4,2026-05-01 00:44:13,140
5,2026-05-01 00:24:35,255
6,2026-05-01 00:06:08,43
7,2026-05-01 00:49:20,164
8,2026-05-01 00:29:38,234
9,2026-05-01 00:39:07,234


In [14]:
rides['pickup_datetime'].describe()

count                       4090836
mean     2026-05-16 10:34:13.127794
min             2008-12-31 23:05:53
25%      2026-05-08 21:15:44.750000
50%             2026-05-16 01:47:51
75%      2026-05-23 15:40:14.250000
max             2026-06-01 00:20:35
Name: pickup_datetime, dtype: object

In [15]:
rides = rides[rides.pickup_datetime >= '2026-05-01']
rides = rides[rides.pickup_datetime < '2026-06-01']
rides['pickup_datetime'].describe()


count                       4090822
mean     2026-05-16 10:38:44.369095
min             2026-05-01 00:00:00
25%             2026-05-08 21:15:48
50%             2026-05-16 01:47:58
75%      2026-05-23 15:40:18.500000
max             2026-05-31 23:59:59
Name: pickup_datetime, dtype: object

In [16]:
rides.to_parquet('../data/transformed/validated_rides_2026_05.parquet')